#**Analyzing AF predictions with MultimerMapper**
In this Colab notebook we will run MultimerMapper for the protein system composed of the trypanosomatid proteins TcMRG, TcMRGBP and TcBDF6. We will understand how to run it, how it analyze AI-derived protein structure predictions and what is its output.

First of all, we need to intall it in the virtual machine:

In [ ]:
# @title Installation

# Downgrade python to 3.10
!apt-get install -y python3.10 python3.10-distutils
!update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

# Install dependencies
!pip install \
    biopython==1.82 \
    pandas==1.5.3 \
    numpy==1.25.0 \
    matplotlib==3.8.0 \
    python-igraph==0.11.3 \
    plotly==5.18.0 \
    ipython==8.20.0 \
    pycairo==1.26.1 \
    cairocffi==1.6.1 \
    scikit-learn==1.3.0 \
    wordcloud==1.9.3 \
    py3dmol==2.4.0 \
    nbformat==5.9.2 \
    colorcet==3.1.0 \
    tqdm==4.66.5 \
    plotnine==0.12.4 \
    seaborn==0.13.2 \
    beautifulsoup4==4.13.3 \
    networkx==3.3 \
    requests

# Get MultimerMapper repository for the course
!git clone https://github.com/elviorodriguez/BiotrAIn_MultimerMapper.git



###MultimerMapper options
MultimerMapper runs from the command line by calling the source code file "multimer_mapper.py" using python. Once installed, we can see the usage by calling it with the option -h.

If you are thinking to install the software in your own machine, DO NOT USE THIS CODE, as it can break your system. Instead, go to the original repository (https://github.com/elviorodriguez/MultimerMapper) and follow the installation instructions.

In [ ]:
# Usage message
!python3.10 /content/BiotrAIn_MultimerMapper/multimer_mapper.py -h

#

---

#Initialize the system (2mers prediction)
All versus all dimeric predictions (2mers) can be computed manually, adding sequences one by one. However, MultimerMapper build this by simply passing the FASTA file of our system with a special format (headers with ```>ID|Symbol``` syntax). In the results directory that we chose (using the ```out_path``` option), we will find the directory ```combinations_suggestions```, which contains different formats to predict our 2-mers. Here, we will be using the JSON file ```ids_af3_jobs_all.json``` that can be uploaded directly to the AF3 server.

In [ ]:
!python3.10 /content/BiotrAIn_MultimerMapper/multimer_mapper.py \
    /content/BiotrAIn_MultimerMapper/course_material/Tc_proteins.fasta \
    --out_path mm_Tc_initialization

#

---

# Analyze the 2-mers

## AF3 to AF2 conversion
MultimerMapper was developed before the release of AF3. Hence, it is necesary to convert AF3 predictions to a format similar to the AF2 output. For example, AF3 gives CIF files instead of PDB files. Also, metrics are provided by ATOM instead of by RESIDUE. The conversion computes the mean pLDDT and mean pAE for each residue and each residue pair, respectively, using the atomic metrics.

In [ ]:
!python3.10 /content/BiotrAIn_MultimerMapper/utils/af3_compatibility.py \
    /content/BiotrAIn_MultimerMapper/course_material/Tc_complex_2mers.zip \
    --out_dir Tc_complex_2mers_converted

## Run 2-mers
Now that we have the predictions in a compatible format, we can analyze our data. With the option ```AF_2mers``` we indicate where the converted 2-mers are located and with ```out_path``` where to save the results. Given that the software was tought to be executed interactively to optimize the results, simultaneously observing the results, in Goolge Colab we need to pass in the following options to skip the interactive sessions:

*   ```auto_domains```: Indentifies domains without semi-automatic optimization
*   ```first_plot```: Gives the first generated plot
*   ```skip_plots```: Avoids displaying plots

On the other hand, we use ```overwrite``` just in case we need to run the block of code again. It will overwrite the output without throwing an error. Be carful while executing long runs.

In [ ]:
!python3.10 /content/BiotrAIn_MultimerMapper/multimer_mapper.py \
    /content/BiotrAIn_MultimerMapper/course_material/Tc_proteins.fasta \
    --AF_2mers /content/Tc_complex_2mers_converted \
    --out_path mm_Tc_2mers \
    --auto_domains \
    --first_plot \
    --overwrite \
    --skip_plots


#

---

# MultimerMapper iterations (N-mers)
Beyond the 2-mers, to fully understand the interactions that govern a molecular system, MultimerMapper analyze how protein behave under different modelling conditions. For this, MultimerMapper will guide us to generate the optimal set of protein structure predictions by combining the proteins optimally. For example, a protein may interact with another protein in one situation, but if we add another protein, the interaction may be inhibited due to a conformational change or due to competition for the same surface. MultimerMapper is aware of this and has internal algorithms that detect these behaviors.

The idea is that in each iteration we compute 3-mers, 4-mers, etc., and we detect which combinations form "stable" complexes, until no more surfaces are available. The biggest stable combinations will be the stoichiometries of the complexes inside of our system of interest.

To continue the iterative process after running the 2-mers, we must search for the combination suggestions (```mm_Tc_2mers/combinations_suggestions/ids_af3_jobs_all.json```), predict them with AF3, convert them to a compatible format and provide them again along the 2-mers to MultimerMapper. The following code does exactly that, using the precomputed N-mers.

In [ ]:
# @title 2+3-mers iteration

# Create directory to store N-mers
!mkdir Tc_complex_Nmers_converted

# Convert 3-mers to compatible format
!python3.10 /content/BiotrAIn_MultimerMapper/utils/af3_compatibility.py \
    /content/BiotrAIn_MultimerMapper/course_material/Tc_complex_3mers.zip \
    --out_dir Tc_complex_Nmers_converted/Tc_complex_3mers_converted

# Run 2+3-mers iteration
!python3.10 /content/BiotrAIn_MultimerMapper/multimer_mapper.py \
    /content/BiotrAIn_MultimerMapper/course_material/Tc_proteins.fasta \
    --AF_2mers /content/Tc_complex_2mers_converted \
    --AF_Nmers /content/Tc_complex_Nmers_converted \
    --out_path mm_Tc_2+3mers \
    --auto_domains \
    --first_plot \
    --overwrite \
    --skip_plots


In [ ]:
# @title 2+3+4-mers iteration (CONVERGENCE point of the system)

# Create directory to store N-mers (-p avoids error if it exists)
!mkdir -p Tc_complex_Nmers_converted

# Convert 4-mers to compatible format
!python3.10 /content/BiotrAIn_MultimerMapper/utils/af3_compatibility.py \
    /content/BiotrAIn_MultimerMapper/course_material/Tc_complex_4mers.zip \
    --out_dir Tc_complex_Nmers_converted/Tc_complex_4mers_converted

# Run 2+3+4-mers iteration
!python3.10 /content/BiotrAIn_MultimerMapper/multimer_mapper.py \
    /content/BiotrAIn_MultimerMapper/course_material/Tc_proteins.fasta \
    --AF_2mers /content/Tc_complex_2mers_converted \
    --AF_Nmers /content/Tc_complex_Nmers_converted \
    --out_path mm_Tc_2+3+4mers \
    --auto_domains \
    --first_plot \
    --overwrite \
    --skip_plots
